# Task 3 — 16-Queens via Genetic Algorithm

**Improvements over original:**
- `Board` stores only a `cols: list[int]` (one int per row) — no `Queen` objects → smaller memory, faster access
- `fitness()` is **cached** on each `Board` instance — never recomputed after creation
- `crossover()` / `mutate()` create new `Board` objects → cache never becomes stale
- `genetic_algorithm` pre-evaluates fitness once per individual per generation using a cached array
- **Elitism**: top 10% of parents are carried forward unchanged — prevents regression
- No confusing `_` variable used as both loop dummy and fitness tracker

## 1. Imports

In [1]:
import random

## 2. `Board` class

**State representation**: a list `cols` of length `N` where `cols[r]` = column of the queen in row `r`. Since we place exactly one queen per row, row conflicts are impossible by construction — we only need to check column and diagonal conflicts.

In [2]:
class Board:
    """A placement of N queens — one per row — represented as a column array.

    Attributes
    ----------
    size : int
    cols : list[int]   cols[r] = column of queen in row r  (0-indexed)
    """

    __slots__ = ('size', 'cols', '_fitness')

    def __init__(self, size: int, cols: list[int] | None = None) -> None:
        self.size = size
        # Accept an explicit column list (for crossover/mutate) or randomize
        self.cols    = cols if cols is not None else [random.randint(0, size - 1)
                                                      for _ in range(size)]
        self._fitness: int | None = None   # computed lazily, then cached

    # ----------------------------------------------------------------- fitness
    def fitness(self) -> int:
        """
        Number of NON-attacking queen pairs.
        Max possible = N*(N-1)//2  (perfect board).

        Computed once and cached — O(N^2) first call, O(1) afterwards.
        """
        if self._fitness is not None:
            return self._fitness

        attacks = 0
        n = self.size
        cols = self.cols
        for i in range(n):
            for j in range(i + 1, n):
                # Same column or same diagonal
                if cols[i] == cols[j] or abs(i - j) == abs(cols[i] - cols[j]):
                    attacks += 1

        self._fitness = (n * (n - 1)) // 2 - attacks
        return self._fitness

    # -------------------------------------------------------------- crossover
    def crossover(self, other: 'Board') -> 'Board':
        """Two-point crossover — returns a NEW Board (cache stays valid on parents)."""
        n  = self.size
        c1 = random.randint(1, n - 2)
        c2 = random.randint(c1 + 1, n - 1)

        child_cols = (
            self.cols[:c1] +
            other.cols[c1:c2] +
            self.cols[c2:]
        )
        return Board(n, child_cols)

    # ----------------------------------------------------------------- mutate
    def mutate(self, mutation_rate: float) -> 'Board':
        """Randomly perturb columns — returns a NEW Board."""
        new_cols = [
            random.randint(0, self.size - 1) if random.random() < mutation_rate else c
            for c in self.cols
        ]
        return Board(self.size, new_cols)

    # ----------------------------------------------------------------- display
    def display(self) -> None:
        """Print the board and queen column positions (1-indexed)."""
        n = self.size
        for r, c in enumerate(self.cols):
            print('. ' * c + 'Q ' + '. ' * (n - c - 1))
        print()
        print('Queen columns (1-indexed):')
        print([c + 1 for c in self.cols])

## 3. Selection

In [3]:
def tournament_select(population: list[Board], k: int = 5) -> Board:
    """Tournament selection: pick k candidates, return the fittest."""
    contestants = random.sample(population, k)
    return max(contestants, key=lambda b: b.fitness())

## 4. Genetic algorithm

**Elitism**: the top `elite_frac` fraction of parents survive unchanged each generation, preventing the best solutions from being lost.

In [4]:
def genetic_algorithm(
    board_size:      int   = 16,
    population_size: int   = 256,
    mutation_rate:   float = 0.2,
    generations:     int   = 500,
    elite_frac:      float = 0.1,
) -> Board:
    """
    Solve the N-queens problem with a genetic algorithm.

    Parameters
    ----------
    board_size      : N (board is N x N)
    population_size : number of individuals
    mutation_rate   : probability of mutating each queen's column
    generations     : maximum number of generations
    elite_frac      : fraction of best parents copied into next generation

    Returns
    -------
    Best Board found
    """
    max_fitness  = (board_size * (board_size - 1)) // 2
    elite_count  = max(1, int(population_size * elite_frac))

    population   = [Board(board_size) for _ in range(population_size)]
    best_seen    = 0   # tracks progress for printing

    for gen in range(generations):
        # -- evaluate & sort (fitness is cached, so sort only calls fitness() once per board)
        population.sort(key=lambda b: b.fitness(), reverse=True)
        best_board   = population[0]
        best_fitness = best_board.fitness()

        # -- report only when fitness improves
        if best_fitness > best_seen:
            best_seen = best_fitness
            print(f'Generation {gen:4d}: best fitness = {best_fitness}/{max_fitness}')

        # -- check for perfect solution
        if best_fitness == max_fitness:
            print('Optimal solution found!')
            return best_board

        # -- build next generation
        # Elites pass through unchanged
        new_population: list[Board] = population[:elite_count]

        while len(new_population) < population_size:
            parent_a = tournament_select(population)
            parent_b = tournament_select(population)
            child    = parent_a.crossover(parent_b)
            if random.random() < mutation_rate:
                child = child.mutate(mutation_rate)
            new_population.append(child)

        population = new_population

    # Return best after all generations
    population.sort(key=lambda b: b.fitness(), reverse=True)
    return population[0]

## 5a. Run — set parameters & solve

In [9]:
# -- parameters (user can adjust) ---------------------------------------------
BOARD_SIZE      = 16
POPULATION_SIZE = 256
GENERATIONS     = 500

mutation_rate_input = input('Enter mutation rate (e.g. 0.2): ').strip()
try:
    MUTATION_RATE = float(mutation_rate_input)
    if not 0.0 <= MUTATION_RATE <= 1.0:
        raise ValueError
except ValueError:
    print('Invalid input, using default 0.2')
    MUTATION_RATE = 0.2

print(f'Running: size={BOARD_SIZE}, pop={POPULATION_SIZE}, '
      f'mutation={MUTATION_RATE}, generations={GENERATIONS}')
print('-' * 55)

solution = genetic_algorithm(
    board_size      = BOARD_SIZE,
    population_size = POPULATION_SIZE,
    mutation_rate   = MUTATION_RATE,
    generations     = GENERATIONS,
)

Enter mutation rate (e.g. 0.2): 0.9
Running: size=16, pop=256, mutation=0.9, generations=500
-------------------------------------------------------
Generation    0: best fitness = 110/120
Generation    1: best fitness = 114/120
Generation   23: best fitness = 115/120
Generation   26: best fitness = 116/120
Generation   87: best fitness = 117/120
Generation  282: best fitness = 118/120
Generation  399: best fitness = 119/120


## 5b. Display result

In [10]:
max_fitness = (BOARD_SIZE * (BOARD_SIZE - 1)) // 2
print('-' * 55)
print(f'Final fitness : {solution.fitness()}/{max_fitness}')
if solution.fitness() == max_fitness:
    print('Status        : OPTIMAL (no queens attack each other)')
else:
    attacks = max_fitness - solution.fitness()
    print(f'Status        : {attacks} attacking pair(s) remaining')
print('-' * 55)
print('Board:')
solution.display()

-------------------------------------------------------
Final fitness : 119/120
Status        : 1 attacking pair(s) remaining
-------------------------------------------------------
Board:
. . . . . . Q . . . . . . . . . 
. . . . . . . . . . . . . . . Q 
. . . . . . . . . . . . Q . . . 
Q . . . . . . . . . . . . . . . 
. . . . . . . Q . . . . . . . . 
. . . . . . . . . . Q . . . . . 
. . Q . . . . . . . . . . . . . 
. . . . . . . . . . . . . . Q . 
. Q . . . . . . . . . . . . . . 
. . . . . . . . . Q . . . . . . 
. . Q . . . . . . . . . . . . . 
. . . . . . . . . . . . . Q . . 
. . . . . . . . . . . Q . . . . 
. . . . Q . . . . . . . . . . . 
. . . . . . . . Q . . . . . . . 
. . . . . Q . . . . . . . . . . 

Queen columns (1-indexed):
[7, 16, 13, 1, 8, 11, 3, 15, 2, 10, 3, 14, 12, 5, 9, 6]
